<div align="center">
  <img src="https://capsule-render.vercel.app/api?type=waving&color=0:0f2027,50:203a43,100:2c5364&height=200&section=header&text=Deep%Learning%20Projet&fontSize=50&fontColor=ffffff&animation=fadeIn&fontAlignY=38&desc=Fall%20Semester%202026/2027&descAlignY=58&descSize=18" width="100%" alt="banner"/>
</div>

Imports

In [ ]:
import pandas as pd
import tensorflow as tf
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from tensorflow.keras import Sequential, regularizers
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D, Input, Rescaling



project_root = Path.cwd().resolve()
if project_root.name == "src":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from utils.modelling_functions import images_to_tensor_batches

Labels Dataset Import

In [ ]:
labels_df = pd.read_csv(project_root / "Data" / "labels_with_hashes.csv")

In [ ]:
labels_df.head()

In [ ]:
directory = project_root / "Data" / "Tiny_ImageNet_Project_2026_Students" / "images"

In [ ]:
image_names = labels_df['filename'].values

In [ ]:
labels_encoded = labels_df['class_id'].astype('category').cat.codes.values
print(labels_encoded)

In [ ]:
image_paths = [os.path.join(directory, fname) for fname in image_names]
print(image_paths[:5])

In [ ]:
image_size = (64, 64)
batch_size = 32

In [ ]:
train_names, temp_names, train_y, temp_y = train_test_split(
    image_names, labels_encoded, test_size=0.3, stratify=labels_encoded, random_state=42)

val_names, test_names, val_y, test_y = train_test_split(
    temp_names, temp_y, test_size=0.5, stratify=temp_y, random_state=42)

In [ ]:
train_ds = images_to_tensor_batches(
    train_names, train_y, directory, batch_size, training=True
)
val_ds = images_to_tensor_batches(val_names, val_y, directory, batch_size)
test_ds = images_to_tensor_batches(test_names, test_y, directory, batch_size)

In [ ]:
for images, labels_batch in train_ds.take(1):
    print(images.shape)

In [ ]:
import matplotlib.pyplot as plt

for images, labels_batch in train_ds.take(1):
    plt.figure(figsize=(8, 8))
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(tf.cast(images[i], tf.uint8))
        plt.title(labels_batch[i].numpy())
        plt.axis('off')
    plt.show()

In [ ]:
model = Sequential()
model.add(Input(shape=(64, 64, 3)))
model.add(Rescaling(1.0 / 255.0))
model.add(Conv2D(32, 3, activation='relu'))
model.add(MaxPooling2D())
model.add(Conv2D(64, 3, activation='relu'))
model.add(MaxPooling2D())
model.add(Flatten())
model.add(Dense(128, kernel_regularizer=regularizers.l2(0.001), activation='relu'))
model.add(Dense(200, activation='softmax'))

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=10)

In [ ]:
plt.plot(history.history["accuracy"], label = "train accuracy")
plt.plot(history.history["val_accuracy"], label = "validation accuracy")
plt.title("Model Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()


In [ ]:
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='validation')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
model.evaluate(test_ds)